# Using PyMilvus's Model To Generate Text Embeddings

使用 PyMilvus（版本高于 2.4.0）的文本嵌入技术，快速提升您的搜索能力。本指南将介绍如何利用 PyMilvus 模型提取丰富的文本嵌入，为强大的搜索功能打下基础。

在本文中，我们将逐一介绍**密集型嵌入模型**、**稀疏型嵌入模型**以及**混合模型**，并展示如何实际应用这些模型。

首先，我们安装所需的依赖项（您也可以使用 `virtualenv` 创建新的环境）：

In [ ]:
#! pip install pymilvus[model]

In [1]:
import os

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'

os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

os.environ['TORCH_HOME'] = CUSTOM_CACHE

对于大多数使用场景，您只需调用 `ef(texts)` 即可生成用于存储或检索的嵌入向量。但当查询和文档需要不同的处理方式时，您可以使用两个特定函数：通过 `encode_documents` 处理文档以生成其嵌入向量，并将这些向量存储在向量数据库中；而在检索时，则使用 `encode_queries` 处理查询以生成其嵌入向量，再利用该向量在数据库中进行搜索。

## Dense Embedding
**密集嵌入**是一种自然语言处理技术，用于将词语或短语表示为高维空间中的连续、密集向量，从而捕捉语义关系。

### OpenAI Embedding Function

OpenAI 提供密集嵌入服务，但用户必须先注册并获取 API 密钥才能使用。在环境变量中正确设置 API 密钥后，即可开始使用 OpenAIEmbeddingFunction 等工具生成密集嵌入。

In [ ]:
docs=[
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]
queries=docs

In [ ]:
from pymilvus import model

# initialize using 'text-embedding-3-large'
openai_ef=model.dense.OpenAIEmbeddingFunction(
    model_name="text-embedding-3-large", # Specify the model name
    dimensions=512, # Set the embedding dimensionality according to MRL feature.
)

# get the embeddings in general way

queries_embeddings=openai_ef(queries)
docs_embeddings=openai_ef(docs)

# get the embeddings in specified way
queries_embeddings=openai_ef.encode_queries(queries)
docs_embeddings=openai_ef.encode_documents(docs)

# now we can check the dimension of embedding from results and the embedding function
print("dense dim: ",openai_ef.dim, queries_embeddings[0].shape)
print("dense dim: ",openai_ef.dim, docs_embeddings[0].shape)

使用 OpenAIEmbeddingFunction 时，`encoding_queries` 和 `encoding_documents` 的操作流程完全相同，因此可以直接使用 `openai_ef(texts)` 替代。

- `openai_ef(texts)`：与其他两个函数功能相同。
- `openai_ef.encode_queries(queries)`：与其他两个函数功能相同。
- `openai_ef.encode_documents(documents)`：与其他两个函数功能相同。

此外，您还可以通过在函数配置中直接提供 OpenAI 官方参数（如 api_key 和 base_url）来初始化 OpenAIEmbeddingFunction。

In [ ]:
# initialize using api_key directly.
openai_ef=model.dense.OpenAIEmbeddingFunction(model_name="text-embedding-3-small", api_key='sk-api-key')
# get the embeddings
queries_embeddings=openai_ef.encode_queries(queries)
docs_embeddings=openai_ef.encode_documents(docs)
print("dense dim: ",openai_ef.dim, queries_embeddings[0].shape)
print("dense dim: ",openai_ef.dim, docs_embeddings[0].shape)

### Sentence Transformer Embedding Function

除了像 OpenAI 这样的托管服务外，还存在多种强大的开源密集嵌入模型。对于这些模型，可以使用基于 Sentence-Transformer 的 SentenceTransformerEmbeddingFunction 来提取文本嵌入。

In [ ]:
docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
    "The Turing Test, proposed by Alan Turing, is a measure of a machine's ability to exhibit intelligent behavior.",
    "Deep learning is a subset of machine learning in artificial intelligence that has networks capable of learning unsupervised from data that is unstructured or unlabeled.",
    "The concept of neural networks, which are vital to deep learning algorithms, was inspired by the understanding of the human brain's structure and function.",
    "Artificial intelligence applications range from natural language processing to expert systems, and from automated reasoning to machine learning.",
    "The development of quantum computing holds the potential to drastically increase the processing power available for artificial intelligence systems.",
    "In the field of robotics, artificial intelligence is used to enable autonomous decision-making by robots in complex environments.",
    "Ethical considerations in AI research and application are becoming increasingly important as the technology advances and becomes more integrated into daily life.",
    "Reinforcement learning, a type of machine learning algorithm, enables an agent to learn in an interactive environment by trial and error using feedback from its own actions and experiences.",
    "AI has the potential to revolutionize industries by optimizing processes, enhancing decision-making, and creating new opportunities for innovation."
]

queries=docs

In [ ]:
from pymilvus import model

sentence_transformer_ef=model.dense.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2",
    device="cuda:0",
)

# get the embeddings in general way

queries_embeddings=sentence_transformer_ef(queries)
docs_embeddings=sentence_transformer_ef(docs)

# get the embeddings in specified way
queries_embeddings=sentence_transformer_ef.encode_queries(queries)
docs_embeddings=sentence_transformer_ef.encode_documents(docs)

print("dense dim: ",sentence_transformer_ef.dim, queries_embeddings[0].shape)
print("dense dim: ",sentence_transformer_ef.dim, docs_embeddings[0].shape)

使用 SentenceTransformerEmbeddingFunction 时，`encoding_queries` 和 `encoding_documents` 将分别在查询和文档前添加 **query_instruction** 和 **doc_instruction**，其余操作保持不变。

- `sentence_transformer_ef(texts)`：不添加任何前缀，直接处理原始文本。
- `sentence_transformer_ef.encode_queries(queries)`：在每个查询前添加 **query_instruction**。
- `sentence_transformer_ef.encode_documents(documents)`：在每个文档前添加 **doc_instruction**。

此外，SentenceTransformerEmbeddingFunction 的初始化可结合 Sentence Transformer 的特性，例如指定 `batch_size` 等参数。某些模型需要在实际文本输入前添加指令。

In [ ]:
# BAAI/bge-small-en-v1.5 建议在生成嵌入时添加指令。
sentence_transformer_ef=model.dense.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-small-en-v1.5",
    device="cuda:0",
    batch_size=8,
    query_instruction="Represent this sentence for searching relevant passages:",
    doc_instruction="Represent this sentence for searching relevant passages:",
)
queries_embeddings=sentence_transformer_ef.encode_queries(queries)
docs_embeddings=sentence_transformer_ef.encode_documents(docs)

print('dense dim: ',sentence_transformer_ef.dim, queries_embeddings[0].shape)
print('dense dim: ',sentence_transformer_ef.dim, docs_embeddings[0].shape)

## Sparse Embedding
**稀疏嵌入**通过向量表示词语或短语，其中大多数元素为零，仅有一个非零元素表示词汇表中某个特定词语的存在。稀疏嵌入模型高效且易于解释，因此适用于对精确词义匹配要求较高的任务。

### Splade Embedding Function

SPLADE嵌入是一种为文档和查询提供高度稀疏表示的模型，它继承了词袋模型（BOW）的优点，例如精确的术语匹配和高效性。我们可以轻松地使用SpladeEmbeddingFunction来调用SPLADE模型。

In [ ]:
docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
    "The Turing Test, proposed by Alan Turing, is a measure of a machine's ability to exhibit intelligent behavior.",
]

queries=docs

版本兼容问题

In [ ]:
# from pymilvus.model.sparse import SpladeEmbeddingFunction
#
# # 默认使用模型名称 naver/splade-cocondenser-ensembledistil。
# # other valid options:
# # - naver/splade_v2_max
# # - naver/splade_v2_distil
# # - naver/splade-cocondenser-selfdistil.
# splade_ef=SpladeEmbeddingFunction()
#
# # get the embeddings in general way.
# queries_embeddings=splade_ef(queries)
# docs_embeddings=splade_ef(docs)
#
# # get the embeddings in specified way.
# queries_embeddings=splade_ef.encode_queries(queries)
# docs_embeddings=splade_ef.encode_documents(docs)
#
# # 由于输出嵌入向量以二维CSR数组格式存储，我们将其转换为列表以便更方便地进行操作。
# print("sparse dim: ",splade_ef.dim, list(queries_embeddings)[0].nnz)
# print("sparse dim: ",splade_ef.dim, list(docs_embeddings)[0].nnz)

使用 SpladeEmbeddingFunction 时，`encoding_queries` 和 `encoding_documents` 会分别在查询和文档前添加 **query_instruction** 和 **doc_instruction**，**k_tokens_query** 用于剪裁查询结果，**k_tokens_document** 用于剪裁文档结果。

- `splade_ef(texts)`：不添加任何前缀，不会对结果进行剪裁。
- `splade_ef.encode_queries(queries)`：在每个查询前添加 **query_instruction**，**k_tokens_query** 用于剪裁查询结果。
- `splade_ef.encode_documents(documents)`：在每个文档前添加 **doc_instruction**，**k_tokens_document** 用于剪裁文档结果。

默认情况下，模型直接输出结果。但在某些情况下，用户可能仅希望保留特定有效值中最大的 k 个值。此时，用户可分别为查询和文档指定参数 'k_tokens_query' 和 'k_tokens_document'。

## BM25 Embedding Function

BM25 是一种用于信息检索的排名函数，用于估计文档与给定搜索查询的相关性。它通过引入文档长度归一化和词频饱和来增强基本的词频方法。BM25 可以将文档表示为词重要性得分向量，从而生成稀疏嵌入，实现对稀疏向量空间中高效检索和排序。

In [6]:
import nltk
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
    nltk.data.find('tokenizers/punkt_tab')
    print("✅ All resources loaded successfully.")
except LookupError as e:
    print("❌ Still missing:", e)

✅ All resources loaded successfully.


In [7]:
from pymilvus.model.sparse.bm25.tokenizers import build_default_analyzer
from pymilvus.model.sparse import BM25EmbeddingFunction

# there are some built-in analyzers for several languages, now we use 'en' for English.
analyzer=build_default_analyzer(language="en")

corpus=[
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]

# analyzer can tokenizer the text into tokens
tokens=analyzer(corpus[0])
print("tokens: ",tokens)

tokens:  ['artifici', 'intellig', 'found', 'academ', 'disciplin', '1956']


BM25 算法通过内置分析器将文本首先分割为词元（token），例如英语中的“artifici”、“intellig”和“academ”。然后，它对这些词元进行统计分析，评估其在文档中的出现频率和分布情况。BM25 的核心是根据每个词元的重要性计算其相关性得分，出现频率较低的词元得分更高。这一简洁的过程能够有效排序文档，使其与查询的相关性得到体现。

因此，我们需要构建一个数据集（或语料库）来获取这些统计信息。

In [18]:
bm25_ef=BM25EmbeddingFunction(analyzer)

# Fit the model on the corpus to get the statstics of the corpus
bm25_ef.fit(corpus)
docs=[
    "The field of artificial intelligence was established as an academic subject in 1956.",
    "Alan Turing was the pioneer in conducting significant research in artificial intelligence.",
    "Originating in Maida Vale, London, Turing grew up in the southern regions of England.",
    "In 1956, artificial intelligence emerged as a scholarly field.",
    "Turing, originally from Maida Vale, London, was brought up in the south of England."
]
queries=docs

# get the embeddings in specified way
queries_embeddings=bm25_ef.encode_queries(queries)
docs_embeddings=bm25_ef.encode_documents(docs)

# Since the output embeddings are in a 2D csr_array format, we convert them to a list for easier manipulation
print("类型差异由 SciPy 版本引起，不影响功能。\ntype(list(queries_embeddings)[0]:",type(list(queries_embeddings)[0]))
print("sparse dim:", bm25_ef.dim, list(queries_embeddings)[0].shape)
print("sparse dim:", bm25_ef.dim, list(docs_embeddings)[0].shape)

类型差异由 SciPy 版本引起，不影响功能。
type(list(queries_embeddings)[0]: <class 'scipy.sparse._csr.csr_array'>
sparse dim: 21 (21,)
sparse dim: 21 (21,)


使用 BM25EmbeddingFunction 时，`encoding_queries` 和 `encoding_documents` 在数学上不可互换。目前没有可用的 `bm25_ef(texts)` 实现。

- `bm25_ef(texts)`：不可用。
- `bm25_ef.encode_queries(queries)`：有独立的实现方式。
- `bm25_ef.encode_documents(documents)`：有独立的实现方式。

每次重新拟合数据都会耗费时间；我们提供保存和加载功能以提高效率。

In [19]:
bm25_ef.save("example.json")

new_bm25_ef=BM25EmbeddingFunction(analyzer)
new_bm25_ef.load("example.json")

queries_embeddings=new_bm25_ef.encode_queries(queries)
docs_embeddings=new_bm25_ef.encode_documents(docs)

print("sparse dim:", bm25_ef.dim, list(queries_embeddings)[0].shape)
print("sparse dim:", bm25_ef.dim, list(docs_embeddings)[0].shape)

sparse dim: 21 (21,)
sparse dim: 21 (21,)


在相似分布中计算统计量对于获得准确结果至关重要。但当缺乏语料库时，我们为MS MARCO数据集提供了专门针对英语的预构建、拟合数据。这将把一个预构建的JSON文件下载到您的本地路径中。

In [20]:
prebuild_bm25_ef=BM25EmbeddingFunction(analyzer)

# load the pre-build json file without fitting the corpus
prebuild_bm25_ef.load()
queries_embeddings=prebuild_bm25_ef.encode_queries(queries)
docs_embeddings=prebuild_bm25_ef.encode_documents(docs)

print("sparse dim:", bm25_ef.dim, list(queries_embeddings)[0].shape)
print("sparse dim:", bm25_ef.dim, list(docs_embeddings)[0].shape)

path is None, using default bm25_msmarco_v1.json.
2026-08-18 09:19:07,461 [INFO][load]: path is None, using default bm25_msmarco_v1.json. (bm25.py:194)
bm25_msmarco_v1.json not found, start downloading from https://github.com/milvus-io/pymilvus-assets/releases/download/v0.1-bm25v1/bm25_msmarco_v1.json to ./bm25_msmarco_v1.json.
2026-08-18 09:19:07,462 [INFO][load]: bm25_msmarco_v1.json not found, start downloading from https://github.com/milvus-io/pymilvus-assets/releases/download/v0.1-bm25v1/bm25_msmarco_v1.json to ./bm25_msmarco_v1.json. (bm25.py:197)
bm25_msmarco_v1.json has been downloaded successfully.
2026-08-18 09:19:23,373 [INFO][load]: bm25_msmarco_v1.json has been downloaded successfully. (bm25.py:204)


sparse dim: 21 (3889385,)
sparse dim: 21 (3889385,)


Finally, let's show analyzers for other languages.

In [22]:
de_text = "Alan Turing war die erste Person, die umfangreiche Forschungen im Bereich der KI durchführte."
fr_text = "Alan Turing était la première personne à mener des recherches approfondies en IA."
zh_text = "艾伦·图灵是第一个进行人工智能领域深入研究的人。"

de_analyzer=build_default_analyzer(language="de")
fr_analyzer=build_default_analyzer(language="fr")
zh_analyzer=build_default_analyzer(language="zh")

de_tokens=de_analyzer(de_text)
fr_tokens=fr_analyzer(fr_text)
zh_tokens=zh_analyzer(zh_text)

print("de_tokens: ",de_tokens)
print("fr_tokens: ",fr_tokens)
print("zh_tokens: ",zh_tokens)

de_tokens:  ['alan', 'turing', 'erst', 'person', 'umfangreich', 'forschung', 'bereich', 'ki', 'durchfuhrt']
fr_tokens:  ['alan', 'turing', 'premi', 'person', 'men', 'recherch', 'approfond', 'ia']
zh_tokens:  ['艾伦', '图灵', '第一个', '人工智能', '领域', '深入研究', '人']


在嵌入模型领域，存在一些混合架构，能够生成密集和稀疏的嵌入。我们将这类模型称为混合模型，并以BGE-M3为例介绍此类模型。

## BGE-M3 Embedding Function

BGE-M3 以其在多语言、多功能和多粒度方面的卓越能力而得名。它能够支持超过 100 种语言，在多语言及跨语言检索任务中树立了新的基准。其独特的功能可在单一框架内实现密集检索、多向量检索和稀疏检索，使其成为多种信息检索（IR）应用的理想选择。

（注意：运行本部分前，请先重启 Jupyter Python 内核。）

In [3]:
docs=[
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
    "The Turing Test, proposed by Alan Turing, is a measure of a machine's ability to exhibit intelligent behavior.",
]

queries=docs

In [4]:
from pymilvus.model.hybrid import BGEM3EmbeddingFunction
import numpy as np

# please set the use_fp16 to False when you are using cpu.
# by default the return options is:
#  return_dense True
#  return_sparse True
#  return_colbert_vecs False
bge_m3_ef=BGEM3EmbeddingFunction(
    model_name='BAAI/bge-m3',
    device="cuda:0",
)

# get the embeddings in general way
queries_embeddings=bge_m3_ef(queries)
docs_embeddings=bge_m3_ef(docs)

# get the embeddings in specified way.
queries_embeddings = bge_m3_ef.encode_queries(queries)
docs_embeddings = bge_m3_ef.encode_documents(docs)

print("dense query dim:",bge_m3_ef.dim["dense"], queries_embeddings["dense"][0].shape)
print("dense document dim:",bge_m3_ef.dim["dense"], docs_embeddings["dense"][0].shape)

# Since the sparse embeddings are in a 2D csr_array format, we convert them to a list for easier manipulation.
print("sparse quey dim:",bge_m3_ef.dim["sparse"], list(queries_embeddings["sparse"])[0].shape)
print("sparse document dim:",bge_m3_ef.dim["sparse"], list(docs_embeddings["sparse"])[0].shape)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

dense query dim: 1024 (1024,)
dense document dim: 1024 (1024,)
sparse quey dim: 250002 (250002,)
sparse document dim: 250002 (250002,)


使用 BGEM3EmbeddingFunction 时，`encoding_queries` 和 `encoding_documents` 的操作流程完全相同，因此可以直接使用 `bge_m3_ef(texts)` 替代。

- `bge_m3_ef(texts)`：与其他两个函数功能相同。
- `bge_m3_ef.encode_queries(queries)`：与其他两个函数功能相同。
- `bge_m3_ef.encode_documents(documents)`：与其他两个函数功能相同。

尽管 BGE-M3 能够同时生成密集和稀疏的嵌入向量，但通过调整返回选项，也可以将其配置为标准的密集或稀疏嵌入生成器。

In [5]:
# use bge-m3 as a dense embeddings
bge_m3_ef=BGEM3EmbeddingFunction(
    model_name='BAAI/bge-m3',
    device="cuda:0",
    return_sparse=False,
)
queries=docs
docs_embeddings=bge_m3_ef.encode_documents(docs)
print("dense docs dim:",bge_m3_ef.dim["dense"], docs_embeddings["dense"][0].shape)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

dense docs dim: 1024 (1024,)


Similarly, we can utilize the BGEM3EmbeddingFunction as a sparse embedding function.

In [6]:
# use bge-m3 as a dense embeddings
bge_m3_ef=BGEM3EmbeddingFunction(
    model_name='BAAI/bge-m3',
    device="cuda:0",
    return_dense=False,
)
queries=docs
docs_embeddings=bge_m3_ef.encode_documents(docs)
print("sparse docs dim:",bge_m3_ef.dim["sparse"], list(docs_embeddings["sparse"])[0].shape)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

sparse docs dim: 250002 (250002,)
